# 🧠 GenAI Student Feedback Generator

In [1]:
# Install packages
%%capture
!pip install streamlit requests pandas

In [2]:
#  CELL 2: Uploading dataset
from google.colab import files
import os

print('Please upload your student_performance_dataset.csv file...')
uploaded = files.upload()

for fname in uploaded.keys():
    os.rename(fname, 'student_performance_dataset.csv')

print('\n✅ Dataset uploaded successfully!')
import pandas as pd
df = pd.read_csv('student_performance_dataset.csv')
print(f'✅ {len(df)} students loaded.')

Please upload your student_performance_dataset.csv file...


Saving student_performance_dataset.csv to student_performance_dataset.csv

✅ Dataset uploaded successfully!
✅ 708 students loaded.


In [5]:
# ── CELL 3: Streamlit
app_code = r"""
import streamlit as st
import pandas as pd
import requests

st.set_page_config(page_title='GenAI Student Feedback', page_icon='🧠', layout='centered')
st.title('🧠 GenAI Student Feedback Generator')
st.caption('Student Report Feedback Analysis · Powered by HuggingFace Mistral-7B')
st.markdown('---')

hf_token = st.sidebar.text_input('🤗 HuggingFace Token', type='password', placeholder='hf_xxxxxxxxxxxx')
if hf_token:
    st.sidebar.success('Token set ✓ Using Mistral-7B')
else:
    st.sidebar.info('No token → rule-based report')

try:
    df = pd.read_csv('student_performance_dataset.csv')
    st.sidebar.success(f'{len(df)} students loaded ✓')
except:
    st.error('Dataset not found!')
    st.stop()

def generate_report(row, tone, audience, token):
    improvement = int(row['Final_Exam_Score']) - int(row['Past_Exam_Scores'])
    trend = ('improved by ' + str(improvement) + ' points' if improvement > 0
             else 'dropped by ' + str(abs(improvement)) + ' points' if improvement < 0
             else 'remained the same')
    if token.strip().startswith('hf_'):
        prompt = (
            '<s>[INST] You are an academic advisor. Write a ' + tone.lower() +
            ' student performance feedback report for a ' + audience.lower() + '.\n\n'
            'Student Details:\n'
            '- Student ID: ' + str(row['Student_ID']) + '\n'
            '- Gender: ' + str(row['Gender']) + '\n'
            '- Study Hours per Week: ' + str(row['Study_Hours_per_Week']) + '\n'
            '- Attendance Rate: ' + str(round(float(row['Attendance_Rate']), 1)) + '%\n'
            '- Past Exam Score: ' + str(row['Past_Exam_Scores']) + '/100\n'
            '- Final Exam Score: ' + str(row['Final_Exam_Score']) + '/100 (score ' + trend + ')\n'
            '- Result: ' + str(row['Pass_Fail']) + '\n'
            '- Parental Education: ' + str(row['Parental_Education_Level']) + '\n'
            '- Internet Access: ' + str(row['Internet_Access_at_Home']) + '\n'
            '- Extracurricular Activities: ' + str(row['Extracurricular_Activities']) + '\n\n'
            'Write: 1) Performance Summary 2) Key Strengths 3) Areas for Improvement 4) Closing.'
            ' Under 250 words. Plain text only. [/INST]'
        )
        try:
            resp = requests.post(
                'https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.2',
                headers={'Authorization': 'Bearer ' + token.strip()},
                json={'inputs': prompt, 'parameters': {'max_new_tokens': 350, 'temperature': 0.7, 'do_sample': True, 'return_full_text': False}},
                timeout=60
            )
            if resp.status_code == 200:
                return resp.json()[0]['generated_text'].strip(), 'mistral'
            return 'HuggingFace API error ' + str(resp.status_code) + ': ' + resp.text[:200], 'error'
        except Exception as e:
            return 'Connection error: ' + str(e), 'error'
    else:
        s = int(row['Final_Exam_Score'])
        att = float(row['Attendance_Rate'])
        hrs = int(row['Study_Hours_per_Week'])
        result = row['Pass_Fail']
        sid = row['Student_ID']
        perf = 'excellent' if s >= 75 else 'satisfactory' if s >= 60 else 'below average'
        att_note = 'strong' if att >= 80 else 'moderate' if att >= 65 else 'low'
        study_note = 'commendable' if hrs >= 25 else 'adequate' if hrs >= 15 else 'insufficient'
        strengths = []
        if att >= 80: strengths.append('Attendance of ' + str(round(att)) + '% shows strong commitment.')
        if hrs >= 25: strengths.append('Studying ' + str(hrs) + ' hours/week shows excellent dedication.')
        if row['Extracurricular_Activities'] == 'Yes': strengths.append('Extracurricular participation shows a well-rounded approach.')
        if not strengths: strengths = ['Shows potential for growth with the right support.']
        improvements = []
        if att < 75: improvements.append('Attendance is ' + str(round(att)) + '% — aim for at least 85%.')
        if hrs < 20: improvements.append('Increase study hours from ' + str(hrs) + ' to at least 20 per week.')
        if s < 60: improvements.append('Practice past exam papers and seek additional tutoring.')
        if not improvements: improvements = ['Continue current habits and challenge yourself further.']
        closing = ('Congratulations on passing! Keep pushing your limits.' if result == 'Pass'
                   else 'With consistent effort and the right support, success is within reach. Believe in yourself!')
        report = (
            'Performance Summary:\n'
            'Student ' + str(sid) + ' has demonstrated ' + perf + ' performance, scoring ' + str(s) + '/100. '
            'Attendance of ' + str(round(att, 1)) + '% has been ' + att_note + ', and weekly study of ' + str(hrs) + ' hours is ' + study_note + '. '
            'The student has ' + result + 'ed this term, with scores having ' + trend + '.\n\n'
            'Key Strengths:\n'
            '• ' + strengths[0] + '\n'
            '• ' + (strengths[1] if len(strengths) > 1 else 'Willingness to engage with the curriculum is evident.') + '\n\n'
            'Areas for Improvement:\n'
            '• ' + improvements[0] + '\n'
            '• ' + (improvements[1] if len(improvements) > 1 else 'Set specific weekly study goals.') + '\n\n'
            'Closing:\n' + closing + '\n\n'
            '— Generated by Student Performance Analytics System'
        )
        return report.strip(), 'rule-based'

col1, col2 = st.columns(2)
with col1:
    selected_id = st.selectbox('Select Student ID', df['Student_ID'].tolist())
    tone = st.radio('Report Tone', ['Encouraging', 'Formal', 'Constructive', 'Motivational'])
with col2:
    audience = st.radio('Report For', ['Student', 'Parent', 'Teacher'])

student = df[df['Student_ID'] == selected_id].iloc[0]

with st.expander('📋 Student Profile', expanded=True):
    c1, c2, c3, c4 = st.columns(4)
    c1.metric('Final Score', str(student['Final_Exam_Score']) + '/100')
    c2.metric('Attendance', str(round(float(student['Attendance_Rate']), 1)) + '%')
    c3.metric('Study Hrs/wk', student['Study_Hours_per_Week'])
    c4.metric('Result', student['Pass_Fail'])
    st.markdown('**Gender:** ' + str(student['Gender']) + ' | **Past Score:** ' + str(student['Past_Exam_Scores']) + ' | **Parent Edu:** ' + str(student['Parental_Education_Level']) + ' | **Internet:** ' + str(student['Internet_Access_at_Home']) + ' | **Extracurricular:** ' + str(student['Extracurricular_Activities']))

if st.button('🚀 Generate Feedback Report', type='primary', use_container_width=True):
    with st.spinner('Generating personalized feedback report...'):
        report, source = generate_report(student.to_dict(), tone, audience, hf_token or '')
    if source == 'error':
        st.error(report)
    else:
        tag = '🤗 Mistral-7B (HuggingFace)' if source == 'mistral' else '📋 Rule-based'
        st.caption('Generated using: ' + tag)
        st.markdown('### 📄 Feedback Report')
        st.text_area('', value=report, height=350, label_visibility='collapsed')
        st.download_button('⬇️ Download Report (.txt)', data=report, file_name='feedback_' + str(selected_id) + '.txt', mime='text/plain')

st.markdown('---')
st.markdown('### ⚡ Batch Report Generation')
batch_n = st.slider('Number of students', 1, 20, 5)
batch_tone = st.selectbox('Batch tone', ['Encouraging', 'Formal', 'Constructive'])
batch_audience = st.selectbox('Batch audience', ['Student', 'Parent', 'Teacher'])

if st.button('⚡ Generate Batch Reports', use_container_width=True):
    sample = df.sample(min(batch_n, len(df)), random_state=42)
    all_reports = []
    progress = st.progress(0)
    status = st.empty()
    for i, (_, row) in enumerate(sample.iterrows()):
        status.text('Generating report ' + str(i+1) + ' of ' + str(len(sample)) + ': ' + str(row['Student_ID']) + '...')
        report, _ = generate_report(row.to_dict(), batch_tone, batch_audience, hf_token or '')
        all_reports.append('=' * 60 + '\nSTUDENT: ' + str(row['Student_ID']) + ' | Result: ' + str(row['Pass_Fail']) + ' | Score: ' + str(row['Final_Exam_Score']) + '/100\n' + '=' * 60 + '\n' + report + '\n\n')
        progress.progress((i + 1) / len(sample))
    status.empty()
    st.success('✅ ' + str(len(all_reports)) + ' reports generated!')
    st.download_button('⬇️ Download All Reports (.txt)', data='\n'.join(all_reports), file_name='batch_feedback_reports.txt', mime='text/plain')
"""

with open('genai_feedback.py', 'w') as f:
    f.write(app_code.strip())

import os
print('✅ genai_feedback.py written!')
print('✅ dataset found!' if os.path.exists('student_performance_dataset.csv') else '❌ dataset missing')

✅ genai_feedback.py written!
✅ dataset found!


In [6]:
#  CELL 4: To Launch the app
import subprocess, time
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)
subprocess.Popen(
    ['streamlit', 'run', 'genai_feedback.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8501)')
print('=' * 50)
print('🧠 GenAI Student Feedback is LIVE!')
print('=' * 50)
print(f'\n👉 Click here to open: {url}\n')
print('Paste your hf_... token in the sidebar for Mistral-7B AI reports')
print('Leave blank for rule-based reports')
print('\n(Keep this cell running to keep the app alive)')

🧠 GenAI Student Feedback is LIVE!

👉 Click here to open: https://8501-m-s-kkb-usw1c0-3fsbim6ltehk3-c.us-west1-0.prod.colab.dev

Paste your hf_... token in the sidebar for Mistral-7B AI reports
Leave blank for rule-based reports

(Keep this cell running to keep the app alive)
